Supervised Training with Checkpoints and Clear Evaluation

In [1]:
import sys
from pathlib import Path
root = Path.cwd()
if (root / 'qcgpt').exists():
    sys.path.insert(0, str(root))
elif (root.parent / 'qcgpt').exists():
    sys.path.insert(0, str(root.parent))


In [2]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from qcgpt.gates import VOCAB, PAD_ID, BOS_CIRC_ID, EOS_CIRC_ID
from qcgpt.models.policy import CircuitPolicy
from qcgpt.training.supervised import build_simplified_dataloader
from qcgpt.data.specs import build_spec_sequence_batch
from qcgpt.encoding import tokens_to_circuit
from qcgpt.evaluation.metrics import quantum_fidelity_from_spec, gate_count
from qcgpt.evaluation.visualize import format_circuit


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vocab_size = len(VOCAB)
model = CircuitPolicy(vocab_size=vocab_size).to(device)
resume_ckpt = 'checkpoints/supervised_current.pt'
if os.path.exists(resume_ckpt):
    state = torch.load(resume_ckpt, map_location=device)
    model.load_state_dict(state['model_state_dict'])
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
train_loader = build_simplified_dataloader(num_samples=10000, batch_size=64, n_qubits=2, raw_max_depth=16)
num_epochs = 100
loss_hist = []
best_loss = float('inf')
os.makedirs('checkpoints', exist_ok=True)
for epoch in range(1, num_epochs + 1):
    total_loss = 0.0
    total_tokens = 0
    for spec_batch, spec_pad_mask, circ_in, circ_tgt in train_loader:
        spec_batch = spec_batch.to(device)
        spec_pad_mask = spec_pad_mask.to(device)
        circ_in = circ_in.to(device)
        circ_tgt = circ_tgt.to(device)
        logits = model(spec_batch, spec_pad_mask, circ_in)
        B, Lc, V = logits.shape
        loss = F.cross_entropy(logits.view(B*Lc, V), circ_tgt.view(B*Lc), ignore_index=PAD_ID)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        num_non_pad = (circ_tgt != PAD_ID).sum().item()
        total_loss += float(loss.item()) * num_non_pad
        total_tokens += num_non_pad
    epoch_loss = total_loss / max(1, total_tokens)
    loss_hist.append(epoch_loss)
    print(f'Epoch {epoch}/{num_epochs}  Loss={epoch_loss:.4f}')
    if epoch % 10 == 0:
        torch.save({'model_state_dict': model.state_dict()}, 'checkpoints/supervised_current.pt')
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save({'model_state_dict': model.state_dict()}, 'checkpoints/supervised_best.pt')
plt.figure(figsize=(6,4))
plt.plot(loss_hist, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Token CE Loss')
plt.title('Supervised Training Loss')
plt.grid(True)
plt.show()
torch.save({'model_state_dict': model.state_dict()}, 'checkpoints/supervised_final.pt')
print('Saved final model to checkpoints/supervised_final.pt')


Evaluation: Teacher vs Prediction and Fidelities

In [6]:
from qcgpt.data.dataset import SimplifiedCircuitDataset
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CircuitPolicy(vocab_size=len(VOCAB)).to(device)
ckpt_paths = [
    'checkpoints/supervised_best.pt',
    'checkpoints/supervised_final.pt',
    'checkpoints/supervised_current.pt',
]
for p in ckpt_paths:
    if os.path.exists(p):
        state = torch.load(p, map_location=device)
        model.load_state_dict(state['model_state_dict'])
        break
model.eval()
ds = SimplifiedCircuitDataset(num_samples=5, n_qubits=2, raw_max_depth=16)
items = [ds[i] for i in range(5)]
for idx, item in enumerate(items):
    teacher_circ = tokens_to_circuit(item['circ_tokens'].tolist())
    print(f'Example {idx+1} teacher:')
    print(format_circuit(teacher_circ))
    spec_tensor = item['spec_tensor'].numpy()
    spec_batch_np, spec_pad_mask_np = build_spec_sequence_batch([spec_tensor])
    spec_b = torch.tensor(spec_batch_np, dtype=torch.float32, device=device)
    pad_b = torch.tensor(spec_pad_mask_np, dtype=torch.bool, device=device)
    with torch.no_grad():
        seqs, logp = model.sample_circuit_tokens(spec_b, pad_b, BOS_CIRC_ID, EOS_CIRC_ID, max_len=64)
    seq = [t for t in seqs[0].tolist() if t != PAD_ID]
    pred_circ = tokens_to_circuit(seq)
    print('Predicted:')
    print(format_circuit(pred_circ))
    fid_teacher = quantum_fidelity_from_spec(spec_tensor, teacher_circ)
    fid_pred = quantum_fidelity_from_spec(spec_tensor, pred_circ)
    print(f'Fidelity teacher: {fid_teacher:.3f} | Fidelity pred: {fid_pred:.3f}')
    for i in range(4):
        print(f'Input Spec used: {spec_tensor[i,0,:,:]}')
        # print(f'Output Spec used: {spec_tensor[i,1,:,:]}')
    print('-'*60)


Example 1 teacher:
00: Y q0
01: S q0
02: CX q1 q0
03: H q0
04: X q1
05: Z q1
Predicted:
00: Y q0
01: S q0
02: H q0
03: X q1
04: CZ q1 q0
Fidelity teacher: 1.000 | Fidelity pred: 0.000
Input Spec used: [[1. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]
Input Spec used: [[0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 0.]]
Input Spec used: [[0. 0.]
 [1. 0.]
 [0. 0.]
 [0. 0.]]
Input Spec used: [[0. 0.]
 [0. 0.]
 [0. 0.]
 [1. 0.]]
------------------------------------------------------------
Example 2 teacher:
00: X q0
01: H q0
02: S q1
03: X q1
04: T q1
05: CZ q1 q0
06: Z q1
Predicted:
00: H q0
01: Z q0
02: Y q1
03: X q1
04: Y q1
05: Z q1
06: X q1
07: T q1
08: Y q1
09: Z q1
10: X q1
11: Z q1
12: S q1
13: CZ q0 q1
Fidelity teacher: 1.000 | Fidelity pred: 0.000
Input Spec used: [[1. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]
Input Spec used: [[0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 0.]]
Input Spec used: [[0. 0.]
 [1. 0.]
 [0. 0.]
 [0. 0.]]
Input Spec used: [[0. 0.]
 [0. 0.]
 [0. 0.]
 [1. 0.]]
-------------------------------------------------